# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Extract and print some basic metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nID: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their fields with @id

def print_record_sets(dataset):
    print("Available Record Sets in the Dataset:")
    for rs in dataset.record_sets:
        print(f"- RecordSet Name: {rs.name}")
        print(f"  RecordSet @id: {rs.id}")
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}) [{field.data_type}]")
        print("-")
        
# Print the overview
print_record_sets(dataset)

### Example Records
Let's view a few records from the first record set.

In [ ]:
# Preview a few records from the first record set (identified by its @id)

# Get all record sets and pick the first one
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets defined in this Croissant dataset.")
else:
    main_record_set = record_sets[0]
    print(f"Previewing records from: {main_record_set.name} (@id: {main_record_set.id})\n")
    counter = 0
    for rec in dataset.records(record_set=main_record_set.id):
        print(json.dumps(rec, indent=2))
        counter += 1
        if counter >= 2:
            break

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from every record set by @id into a pandas DataFrame
dataframes = {}
for rs in dataset.record_sets:
    print(f"Loading records from RecordSet '{rs.name}' (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Columns: {list(df.columns)}\n  Rows: {len(df)}\n")

# Choose main record set for the rest of the notebook
main_record_set_id = dataset.record_sets[0].id if dataset.record_sets else None
if main_record_set_id:
    print(f"Fields in main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All fields are referenced using their `@id`.

In [ ]:
# ----
# You should replace the field @ids below with valid ones seen in the Data Overview section. E.g. from record set fields:
#   - cr:field:coverage_percent
#   - cr:field:peptide_count
#   - cr:field:protein_abundance
#   - cr:field:sample_label
# Here, we select some plausible numeric/grouping fields by @id (please adjust to actual if you see them in Data Overview output).
# ----

main_df = dataframes[main_record_set_id]

# Select a numeric field by @id. Example: 'cr:field:coverage_percent' (must match exact @id from Data Overview)
numeric_field_id = None

# Try to auto-detect a numeric field
possible_numeric_ids = [c for c in main_df.columns if 'percent' in c or 'abundance' in c or 'MW' in c or 'count' in c]
print(f"Auto-detected numeric fields: {possible_numeric_ids}")
if possible_numeric_ids:
    numeric_field_id = possible_numeric_ids[0]

if numeric_field_id is None:
    print("No numeric field detected; please update the code with a field's @id from section 2.")
else:
    # Filter records with value > threshold
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Try grouping by a categorical/group field (e.g. 'cr:field:sample_label')
    group_field_id = None
    group_candidates = [c for c in main_df.columns if 'sample' in c or 'label' in c]
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Grouping by field: {group_field_id}")
    else:
        print("No obvious group field found in DataFrame columns.")

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between fields. Here we plot the normalized numeric field by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and group_field_id:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id + "_normalized", data=filtered_df)
    plt.title(f"Normalized '{numeric_field_id}' by '{group_field_id}'")
    plt.xticks(rotation=45)
    plt.show()
elif numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field_id + "_normalized"], bins=30, kde=True)
    plt.title(f"Distribution of normalized '{numeric_field_id}'")
    plt.show()
else:
    print("No suitable fields selected for visualization. Please check earlier steps.")

## 6. Conclusion
We explored the FAIR² dataset using `mlcroissant`, previewed the structure using entity `@id`s, and performed EDA including normalization and visualization of a main numeric field across groups. You can adapt this notebook to reference additional fields or record sets by their `@id` as given in section 2.